# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Running in Colab')
    !pip install -q networkx faker pandas matplotlib
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

# Graph Neural Networks (GNN) In-Depth

This notebook demonstrates how to build a Graph Convolutional Network (GCN) from scratch using PyTorch to perform node classification on a synthetic Academic Citation Network.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from faker import Faker
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

## 1. Generate Synthetic Citation Network

We use `Faker` to generate synthetic papers with topics (Computer Science, Biology, Physics) and `networkx` to create a citation graph where papers are more likely to cite papers within the same field (homophily).

In [ ]:
fake = Faker()
np.random.seed(42)
torch.manual_seed(42)

# Parameters
NUM_NODES = 200
TOPICS = ['Computer Science', 'Biology', 'Physics']

# Generate Nodes (Papers)
nodes = []
labels = []

for i in range(NUM_NODES):
    topic = np.random.choice(TOPICS)
    nodes.append({
        'id': i,
        'title': fake.sentence(nb_words=4),
        'topic': topic
    })
    labels.append(topic)

df_nodes = pd.DataFrame(nodes)

# Encode Labels
le = LabelEncoder()
y = torch.tensor(le.fit_transform(labels), dtype=torch.long)

# Generate Edges (Citations) with Homophily
G = nx.Graph()
G.add_nodes_from(df_nodes['id'])

for i in range(NUM_NODES):
    # Connect to 2-5 other nodes
    n_edges = np.random.randint(2, 6)
    potential_targets = df_nodes[df_nodes['id'] != i]
    
    for _ in range(n_edges):
        target = potential_targets.sample(1).iloc[0]
        # Higher probability to link if same topic
        prob = 0.8 if target['topic'] == df_nodes.iloc[i]['topic'] else 0.1
        
        if np.random.rand() < prob:
            G.add_edge(i, target['id'])

print(f"Graph generated with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

# Visualize a subgraph
subgraph = G.subgraph(list(range(50)))
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(subgraph, seed=42)
colors = [y[n] for n in subgraph.nodes()]
nx.draw(subgraph, pos, node_color=colors, cmap=plt.cm.Set1, node_size=100, alpha=0.8)
plt.title("Citation Network Subgraph (Colors = Topics)")
plt.show()

## 2. Prepare Features and Adjacency Matrix

For simplicity, we will use an identity matrix as node features (one-hot encoding of node IDs), meaning the model learns purely from structure. We also need the normalized adjacency matrix $\hat{A}$ for the GCN layer propagation rule: $H^{(l+1)} = \sigma(\hat{A} H^{(l)} W^{(l)})$.

Where $\hat{A} = D^{-\frac{1}{2}} (A + I) D^{-\frac{1}{2}}$.

In [ ]:
# Features (Identity Matrix for simplicity)
X = torch.eye(NUM_NODES)

# Adjacency Matrix A
A = nx.adjacency_matrix(G).todense()
A = torch.tensor(A, dtype=torch.float32)

# Add Self-Loops (A + I)
I = torch.eye(NUM_NODES)
A_tilde = A + I

# Degree Matrix D
D = torch.diag(torch.sum(A_tilde, dim=1))

# D^(-1/2)
D_inv_sqrt = torch.pow(D, -0.5)
D_inv_sqrt[torch.isinf(D_inv_sqrt)] = 0

# Normalized Adjacency Matrix: D^(-1/2) * (A + I) * D^(-1/2)
A_hat = torch.mm(torch.mm(D_inv_sqrt, A_tilde), D_inv_sqrt)

print("Normalized Adjacency Matrix shape:", A_hat.shape)

## 3. Implement Graph Convolutional Network (GCN)

We implement a standard 2-layer GCN.

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super(GCNLayer, self).__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, X, A_hat):
        # X_transformed = X * W
        X_transformed = self.linear(X)
        # Propagate: A_hat * X_transformed
        out = torch.mm(A_hat, X_transformed)
        return out

class GCN(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super(GCN, self).__init__()
        self.gcn1 = GCNLayer(in_features, hidden_features)
        self.gcn2 = GCNLayer(hidden_features, out_features)
        self.dropout = nn.Dropout(0.5)

    def forward(self, X, A_hat):
        x = F.relu(self.gcn1(X, A_hat))
        x = self.dropout(x)
        x = self.gcn2(x, A_hat)
        return F.log_softmax(x, dim=1)

# Initialize Model
model = GCN(in_features=NUM_NODES, hidden_features=16, out_features=len(TOPICS))
print(model)

## 4. Training Loop

We split the nodes into training (mask) and testing sets.

In [ ]:
# Masks
train_mask = torch.zeros(NUM_NODES, dtype=torch.bool)
test_mask = torch.zeros(NUM_NODES, dtype=torch.bool)

# 60% Train, 40% Test
indices = torch.randperm(NUM_NODES)
split_idx = int(0.6 * NUM_NODES)
train_mask[indices[:split_idx]] = True
test_mask[indices[split_idx:]] = True

# Optimization
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = nn.NLLLoss()

# Training
losses = []
model.train()
for epoch in range(200):
    optimizer.zero_grad()
    out = model(X, A_hat)
    loss = criterion(out[train_mask], y[train_mask])
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    
    if epoch % 20 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

plt.plot(losses)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

## 5. Evaluation

In [ ]:
model.eval()
_, pred = model(X, A_hat).max(dim=1)

correct = float(pred[test_mask].eq(y[test_mask]).sum().item())
acc = correct / test_mask.sum().item()
print(f'Test Accuracy: {acc:.4f}')

# Compare with random guess baseline
print(f'Baseline Accuracy: {1/len(TOPICS):.4f}')